# ចម្លើយលំហាត់ ជំពូក ៧

> **ចំណាំអំពីប្រភព និងអាជ្ញាបណ្ណ** — នេះជា *companion notebook* របស់សហគមន៍ខ្មែរ។
> កូដទាំងអស់ក្នុង notebook នេះគឺ **ដូចគ្នា ១០០%** ទៅនឹង notebook ផ្លូវការរបស់ឃ្លាំង
> [rasbt/LLMs-from-scratch](https://github.com/rasbt/LLMs-from-scratch/blob/main/ch07/01_main-chapter-code/exercise-solutions.ipynb) (អាជ្ញាបណ្ណ Apache-2.0, (c) Sebastian Raschka)។
> ចំណែកអត្ថបទពន្យល់ជាភាសាខ្មែរគឺជាអត្ថបទដើមរបស់សហគមន៍ (CC BY 4.0) —
> **មិនមែន**ជាការបកប្រែអត្ថបទក្នុងសៀវភៅ *Build a Large Language Model (From Scratch)* ទេ
> ព្រោះអត្ថបទសៀវភៅជាកម្មសិទ្ធិរបស់ Manning Publications និងស្ថិតក្រៅវិសាលភាពអាជ្ញាបណ្ណបើកចំហ។
> សម្រាប់ការពន្យល់ពេញលេញ សូមអានសៀវភៅផ្លូវការ។

**កូដ៖** តក្កវិទ្យាកូដ (logic) ដូច upstream ទាំងស្រុង — មានតែ *comments* ក្នុងកូដដែលត្រូវបានបកប្រែជាភាសាខ្មែរដោយសហគមន៍; កូដដែល comment ជាប់ (commented-out code) នៅដដែល។

**របៀបដំណើរការ៖** ដាក់ notebook នេះក្នុងថតជំពូកពាក់ព័ន្របស់ឃ្លាំង upstream
(ឬចម្លងឯកសារជំនួយដែលកូដត្រូវការ ដូចជា `previous_chapters.py`, `gpt_download.py`
និងឯកសារទិន្នន័យ) រួចដំឡើង dependencies តាម `requirements.txt` ឬប្រើ Google Colab។

## មាតិកា

- ## លំហាត់ ៧.១៖ ការប្តូររចនាបទ prompt
- ## ការរៀបចំ batch និង masking (-100)
- ## លំហាត់ ៧.៤៖ Parameter-efficient finetuning ជាមួយ LoRA

## លំហាត់ ៧.១៖ ការប្តូររចនាបទ prompt

សាកល្បងទ្រង់ទ្រាយ format_input ផ្សេងគ្នា ហើយប្រៀបធៀបលទ្ធផលបង្កើត។

In [ ]:
def format_input(entry):
    instruction_text = (
        f"<|user|>\n{entry['instruction']}"
    )

    input_text = f"\n{entry['input']}" if entry["input"] else ""

    return instruction_text + input_text

In [ ]:
sample_data = [
    {'instruction': 'Identify the correct spelling of the following word.', 'input': 'Ocassion', 'output': "The correct spelling is 'Occasion.'"}, 
    {'instruction': "What is an antonym of 'complicated'?", 'input': '', 'output': "An antonym of 'complicated' is 'simple'."}
]

print(format_input(sample_data[0]))
print()
print(format_input(sample_data[1]))

In [ ]:
# អនុគមន៍ `format_input` នេះត្រូវបានចម្លងពីកូដជំពូក ៧ ដើម

def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )

    input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""

    return instruction_text + input_text

## ការរៀបចំ batch និង masking (-100)

កែ collate function និងការ mask targets ដើម្បីបំពេញលក្ខខណ្ឌលំហាត់។

In [ ]:
import torch
from torch.utils.data import Dataset


class InstructionDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data

        ##########################################################################################
        # ថ្មី៖ បញ្ជីដាច់ដោយឡែកសម្រាប់ប្រវែងការណែនាំ
        self.instruction_lengths = []
        ##########################################################################################
        
        self.encoded_texts = []
        
        for entry in data:
            instruction_plus_input = format_input(entry)
            response_text = f"\n\n### Response:\n{entry['output']}"
            full_text = instruction_plus_input + response_text
            
            self.encoded_texts.append(
                tokenizer.encode(full_text)
            )

            ##########################################################################################
            # ថ្មី៖ ប្រមូលប្រវែងការណែនាំ
            instruction_length = len(tokenizer.encode(instruction_plus_input))
            self.instruction_lengths.append(instruction_length)
            ##########################################################################################
            
    def __getitem__(self, index):
        # ថ្មី៖ ត្រឡប់ទាំងប្រវែងការណែនាំ និងអត្ថបទដាច់ដោយឡែកពីគ្នា
        return self.instruction_lengths[index], self.encoded_texts[index]

    def __len__(self):
        return len(self.data)

In [ ]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

In [ ]:
def custom_collate_fn(
    batch,
    pad_token_id=50256,
    ignore_index=-100,
    allowed_max_length=None,
    device="cpu"
):
    # រកបន្សំវែងបំផុតក្នុង batch
    batch_max_length = max(len(item)+1 for instruction_length, item in batch)  # ថ្មី៖ batch ឥឡូវជា tuple

    # បំពេញ និងរៀបចំ inputs និង targets
    inputs_lst, targets_lst = [], []

    for instruction_length, item in batch:  # ថ្មី៖ batch ឥឡូវជា tuple
        new_item = item.copy()
        # បន្ថែម token <|endoftext|>
        new_item += [pad_token_id]
        # បំពេញបន្សំឱ្យដល់ max_length
        padded = new_item + [pad_token_id] * (batch_max_length - len(new_item))
        inputs = torch.tensor(padded[:-1])  # កាត់ token ចុងក្រោយចេញពី inputs
        targets = torch.tensor(padded[1:])  # រំកិល +1 ទៅស្តាំសម្រាប់ targets

        # ជំនួស tokens បំពេញទាំងអស់ លើកលែងតែទីមួយ ក្នុង targets ដោយ ignore_index
        mask = targets == pad_token_id
        indices = torch.nonzero(mask).squeeze()
        if indices.numel() > 1:
            targets[indices[1:]] = ignore_index

        ##########################################################################################
        # ថ្មី៖ mask tokens បញ្ចូល និងការណែនាំទាំងអស់ក្នុង targets
        targets[:instruction_length-1] = -100
        ##########################################################################################
        
        # កាត់ជាជម្រើសទៅប្រវែងបន្សំអតិបរមា
        if allowed_max_length is not None:
            inputs = inputs[:allowed_max_length]
            targets = targets[:allowed_max_length]
        
        inputs_lst.append(inputs)
        targets_lst.append(targets)

    # បំលែងបញ្ជី inputs និង targets ទៅជា tensors ហើយផ្ទេរទៅ device គោលដៅ
    inputs_tensor = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(targets_lst).to(device)

    return inputs_tensor, targets_tensor

In [ ]:
sample_data = [
    {'instruction': "What is an antonym of 'complicated'?", 'input': '', 'output': "An antonym of 'complicated' is 'simple'."},
    {'instruction': 'Sort the following list in alphabetical order.', 'input': 'Zebra, Elephant, Crocodile', 'output': 'Crocodile, Elephant, Zebra'},
    {'instruction': 'Arrange the given numbers in descending order.', 'input': '5, 12, 8, 3, 15', 'output': '15, 12, 8, 5, 3.'}
]

In [ ]:
from torch.utils.data import DataLoader

train_dataset = InstructionDataset(sample_data, tokenizer)
train_loader = DataLoader(
    train_dataset,
    batch_size=len(sample_data),
    collate_fn=custom_collate_fn,
    num_workers=0
)

In [ ]:
print("Train loader:")
for inputs, targets in train_loader:
    print(inputs.shape, targets.shape)

In [ ]:
print("Inputs:\n", inputs[1])
print("\n\nTargets:\n", targets[1])

In [ ]:
print(tokenizer.decode(list(inputs[1])))

## លំហាត់ ៧.៤៖ Parameter-efficient finetuning ជាមួយ LoRA

ចម្លើយលំហាត់នេះជាគោលគំនិត — សូមមើលភាគបន្ថែម E សម្រាប់កូដ LoRA ពេញលេញ។

In [ ]:
non_masked_targets = targets[1][targets[1] != -100]

print(tokenizer.decode(list(non_masked_targets)))

---

## អាជ្ញាបណ្ណសង្ខេប

- កូដ៖ Apache-2.0 — (c) Sebastian Raschka, គម្រោង [LLMs-from-scratch](https://github.com/rasbt/LLMs-from-scratch) (មិនបានកែប្រែ)
- អត្ថបទពន្យល់ខ្មែរ៖ CC BY 4.0 — សហគមន៍ឯកសារខ្មែរ
- ឯកសារសៀវភៅ និងរូបភាពពាក់ព័ន្ធ៖ កម្មសិទ្ធិ Manning Publications — មិនត្រូវបានចម្លងមកទីនេះទេ